In [5]:
# ============================================================
# TASK 7: TEXT SUMMARIZATION USING PRE-TRAINED MODEL (BART)
# Dataset: CNN-DailyMail (train.csv, validation.csv, test.csv)
# ============================================================

# ----------------------------
# 1️⃣ INSTALL REQUIRED PACKAGES
# ----------------------------
!pip install transformers datasets rouge-score evaluate pandas torch --quiet

# ----------------------------
# 2️⃣ IMPORT LIBRARIES
# ----------------------------
import pandas as pd
import torch
from transformers import BartTokenizer, BartForConditionalGeneration
from rouge_score import rouge_scorer
import evaluate
from tqdm import tqdm
import os

# ----------------------------
# 3️⃣ LOAD DATASET
# ----------------------------
# Upload your CSV files in Colab before running this

# Check if files exist before trying to load
required_files = ["train.csv", "validation.csv", "test.csv"]
for f in required_files:
    if not os.path.exists(f):
        raise FileNotFoundError(f"File not found: {f}. Please upload {f} to your Colab environment.")

try:
    train_df = pd.read_csv("train.csv")
    val_df = pd.read_csv("validation.csv")
    test_df = pd.read_csv("test.csv")
except FileNotFoundError as e:
    print(e)
    # Exit or handle the error appropriately if files are not found
    # For now, we'll re-raise to stop execution if files are missing
    raise

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Check column names
print("Columns:", train_df.columns)

# CNN-DailyMail usually has columns:
# 'article' and 'highlights'
# If different, adjust column names below

# ----------------------------
# 4️⃣ LOAD PRE-TRAINED MODEL (BART)
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

# ----------------------------
# 5️⃣ GENERATE SUMMARIES
# ----------------------------
def generate_summary(text):

    inputs = tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        min_length=40,
        length_penalty=2.0,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# For faster evaluation, we take small subset
test_sample = test_df.head(100)

generated_summaries = []
reference_summaries = []

print("Generating summaries...")

for index, row in tqdm(test_sample.iterrows(), total=len(test_sample)):
    article = str(row["article"])
    reference = str(row["highlights"])

    summary = generate_summary(article)

    generated_summaries.append(summary)
    reference_summaries.append(reference)

# ----------------------------
# 6️⃣ EVALUATE USING ROUGE
# ----------------------------
print("\nCalculating ROUGE scores...")

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

print("\nROUGE Scores:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

# ----------------------------
# 7️⃣ SHOW SAMPLE OUTPUT
# ----------------------------
print("\n================ SAMPLE RESULT ================")
print("Original Article:\n", test_sample.iloc[0]["article"])
print("\nReference Summary:\n", reference_summaries[0])
print("\nGenerated Summary:\n", generated_summaries[0])


Train shape: (287113, 3)
Validation shape: (13368, 3)
Test shape: (11490, 3)
Columns: Index(['id', 'article', 'highlights'], dtype='object')


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Generating summaries...


100%|██████████| 100/100 [02:47<00:00,  1.67s/it]



Calculating ROUGE scores...

ROUGE Scores:
rouge1: 0.4149
rouge2: 0.1947
rougeL: 0.2932
rougeLsum: 0.3546

================ SAMPLE RESULT ================
Original Article:
 Ever noticed how plane seats appear to be getting smaller and smaller? With increasing numbers of people taking to the skies, some experts are questioning if having such packed out planes is putting passengers at risk. They say that the shrinking space on aeroplanes is not only uncomfortable - it's putting our health and safety in danger. More than squabbling over the arm rest, shrinking space on planes putting our health and safety in danger? This week, a U.S consumer advisory group set up by the Department of Transportation said at a public hearing that while the government is happy to set standards for animals flying on planes, it doesn't stipulate a minimum amount of space for humans. 'In a world where animals have more rights to space and food than humans,' said Charlie Leocha, consumer representative on the 

In [6]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import numpy as np
from rouge_score import rouge_scorer
import warnings
import os

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def load_data():
    """Load CNN/DM data - ADJUST YOUR FILE PATHS"""
    print("Loading datasets...")

    # Ensure files exist before reading
    for f in ["train.csv", "validation.csv", "test.csv"]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Please upload {f} to the environment.")

    # Loading subsets for speed
    train_df = pd.read_csv("train.csv").head(1000)
    val_df = pd.read_csv("validation.csv").head(200)
    test_df = pd.read_csv("test.csv").head(50)

    # Fix columns and clean text
    for df in [train_df, val_df, test_df]:
        df.columns = ['id', 'article', 'highlights'][:len(df.columns)]
        df['article'] = df['article'].astype(str).str.replace(r'[\n\r]+', ' ', regex=True)
        df['highlights'] = df['highlights'].astype(str).str.replace(r'[\n\r]+', ' ', regex=True)

    print(f"✓ Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    return train_df, val_df, test_df

def preprocess_function(examples, tokenizer):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(examples["highlights"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = [scorer.score(t, p) for t, p in zip(decoded_labels, decoded_preds)]

    return {
        'rouge1': np.mean([s.rouge1.fmeasure for s in scores]),
        'rouge2': np.mean([s.rouge2.fmeasure for s in scores]),
        'rougeL': np.mean([s.rougeL.fmeasure for s in scores]),
    }

# MAIN EXECUTION
print("🚀 CNN/DailyMail Summarization with T5")
train_df, val_df, test_df = load_data()

# Load model and tokenizer
model_ckpt = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)
model.to(device)

# Preprocess datasets
train_dataset = Dataset.from_pandas(train_df[["article", "highlights"]]).map(
    lambda x: preprocess_function(x, tokenizer), batched=True, remove_columns=["article", "highlights"]
)
eval_dataset = Dataset.from_pandas(val_df[["article", "highlights"]]).map(
    lambda x: preprocess_function(x, tokenizer), batched=True, remove_columns=["article", "highlights"]
)

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🎯 Starting training...")
trainer.train()

# Save model
trainer.save_model("./t5-summarizer-final")

print("\n🧪 Testing inference...")
test_model = AutoModelForSeq2SeqLM.from_pretrained("./t5-summarizer-final").to(device)
test_model.eval()

# Test samples
for i, row in test_df.head(3).iterrows():
    inputs = tokenizer(
        "summarize: " + row['article'][:500],
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = test_model.generate(
            inputs.input_ids,
            max_new_tokens=64,
            num_beams=4,
            do_sample=False
        )

    pred_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n--- Sample {i+1} ---")
    print(f"Input:  {row['article'][:100]}...")
    print(f"Pred:   {pred_summary}")
    print(f"True:   {row['highlights'][:100]}...")
    print("-" * 50)

print("\n✅ SUCCESS! Model saved to ./t5-summarizer-final")

Using device: cuda
🚀 CNN/DailyMail Summarization with T5
Loading datasets...
✓ Train: 1000, Val: 200, Test: 50


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🧪 Testing inference...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


--- Sample 1 ---
Input:  Ever noticed how plane seats appear to be getting smaller and smaller? With increasing numbers of pe...
Pred:   Some experts are questioning if having such packed out planes is putting our health and safety in danger. This week, a U.S consumer advisory group set up by tv. This week, a U.S consumer advisory group set up by tv.
True:   Experts question if  packed out planes are putting passengers at risk . U.S consumer advisory group ...
--------------------------------------------------

--- Sample 2 ---
Input:  A drunk teenage boy had to be rescued by security after jumping into a lions' enclosure at a zoo in ...
Pred:   Rahul Kumar, 17, climbed into the lions' enclosure at a zoo in Ahmedabad. He shouted he would 'kill them' and 'thought I'd stand a good chance'
True:   Drunk teenage boy climbed into lion enclosure at zoo in west India . Rahul Kumar, 17, ran towards an...
--------------------------------------------------

--- Sample 3 ---
Input:  Dougie Freed